In [ ]:
import pandas as pd
import numpy as np
import ast
import re

# 1. Daten einlesen

In [ ]:
# Behandlungsarten importieren
typ = pd.read_csv (
    "project_files/typ.csv",
    sep = ";",
    encoding = "latin1"
)
typ

In [ ]:
# Arztliste importieren
doc = pd.read_csv (
    "project_files/doc.csv",
    sep = ";",
    encoding = "UTF-8",
    header = None,
    names=["arzt_id", "stadt", "fachrichtung"]
)
doc

In [ ]:
#Bewertungen importieren
rev = pd.read_csv(
    "project_files/rev.csv",
    sep=";",
    encoding="UTF-8"
)
rev

# 2.0. Fehlendes überblick

In [ ]:
print((doc.isna().sum()))
print((rev.isna().sum()))
print((typ.isna().sum()))

# 2.1 Doc vervollständigen

In [ ]:
# Fachrichtungen vervollständigen
doc["fachrichtung"] = doc["fachrichtung"].fillna("unbekannt")
print((doc.isna().sum()))

In [ ]:
#Überprüfung doppelter arztids
doc["arzt_id"].duplicated().sum()

# 2.2 Rev vervollständigen

In [ ]:
#Datentypen
print(rev.dtypes)

In [ ]:
# Privat kasse füllen
rev["kasse_privat"] = rev["kasse_privat"].fillna(3).astype(int)

# Kontrolle
print(rev["kasse_privat"].dtype)
print(rev.isna().sum())

In [ ]:
rev_sorted = rev.sort_values(
    by="kasse_privat",
    ascending=True   
)

print(rev_sorted)

In [ ]:
# Doppelte Bewertungen löschen 

# doppelte daten sehen
duplicates = rev[rev.duplicated(subset=['b_id', 'ref_id'], keep=False)]
print(duplicates)

#dropping point
rev_cleaned = rev_sorted.drop_duplicates(subset=['b_id', 'ref_id'], keep="first")

# sorting point
rev_cleaned = rev_cleaned.sort_values(
    by=['ref_id', 'b_id'],
    ascending=True   
)

#wie viele gibt es jetzt
rev_row_before = len(rev)
rev_row_after = len(rev_cleaned) 

#komplette csv auch mit kasse 
rev_cleaned.to_csv(
    "project_files/rev_cleaned.csv",
    sep=";",
    index=False,
    encoding="utf-8"
)

print(f"Vorher Bewertungen: {rev_row_before}")
print(f"Nachher Bewertungen: {rev_row_after}")

In [ ]:
# Die Gesamt Noten vrebessern
# "-" als fehlenden Wert interpretieren
rev_cleaned["gesamt_note"] = rev_cleaned["gesamt_note"].replace("-", np.nan)

rev_cleaned["gesamt_note"] = pd.to_numeric(rev_cleaned["gesamt_note"], errors="coerce")

rev_cleaned.to_csv(
    "project_files/rev_cleaned.csv",
    sep=";",
    index=False,
    encoding="utf-8"
)

print("CSV erfolgreich aktualisiert.")
print(rev_cleaned.dtypes)
print(rev_cleaned["gesamt_note"].isna().sum())

In [ ]:
# gesamtnote löschen, wo man nicht berechnen kann
rows_before = len(rev_cleaned)

rev_cleaned = rev_cleaned[
    # zum löschen von object
    ~(
        rev_cleaned["details"].isna() &
        rev_cleaned["gesamt_note"].isna()
    )
]

print(f"Geloeschte Zeilen: {rows_before - len(rev_cleaned)}")
print(rev_cleaned["gesamt_note"].isna().sum())
print(rev_cleaned["details"].isna().sum())

## Auseinandernehmen von Details und beerechnung

In [ ]:
# Löschen von details, auseinander setzen der spalte und berechnung fehlender gesamtnoten

typ_map = dict(
    zip(
        typ["typ"].astype(str),
        typ["typ_id"].astype(str).str.zfill(2)
    )
)

# -------------------------------------------------
# 2) Sicheres Parsing der details-Spalte
# -------------------------------------------------
def safe_parse(x):
    if x is None or isinstance(x, float):
        return []

    if isinstance(x, list):
        return x

    if isinstance(x, dict):
        return [x]

    if isinstance(x, str):
        x = x.strip()
        if x == "" or x.lower() == "nan":
            return []
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []

    return []

# nur parsen, wenn details noch existiert
if "details" in rev_cleaned.columns:
    rev_cleaned["details"] = rev_cleaned["details"].apply(safe_parse)

# -------------------------------------------------
# 3) Normalisierung einzelner Bewertungen
# -------------------------------------------------
def normalize(e):
    typ_val = e.get("typ")
    note = e.get("note")

    if typ_val is None or note is None:
        return None

    typ_val = str(typ_val)

    if not typ_val.isdigit():
        typ_val = typ_map.get(typ_val)

    if typ_val is None:
        return None

    return {
        "typ": typ_val.zfill(2),
        "note": float(note)
    }

# -------------------------------------------------
# 4) Details -> typ_XX-Spalten auflösen
# -------------------------------------------------
if "details" in rev_cleaned.columns:

    rev_cleaned["bewertungen"] = rev_cleaned["details"].apply(
        lambda lst: [n for e in lst if (n := normalize(e)) is not None]
    )

    expanded = rev_cleaned["bewertungen"].apply(
        lambda lst: {f"typ_{e['typ']}": e["note"] for e in lst}
    ).apply(pd.Series)

    rev_cleaned = pd.concat(
        [
            rev_cleaned.drop(columns=["details", "bewertungen"], errors="ignore"),
            expanded
        ],
        axis=1
    )

# -------------------------------------------------
# 5) gesamt_note nur dort berechnen, wo sie fehlt
# -------------------------------------------------
typ_cols = [c for c in rev_cleaned.columns if c.startswith("typ_")]

if "gesamt_note" not in rev_cleaned.columns:
    rev_cleaned["gesamt_note"] = pd.NA

mask = rev_cleaned["gesamt_note"].isna()
rev_cleaned.loc[mask, "gesamt_note"] = (
    rev_cleaned.loc[mask, typ_cols].mean(axis=1)
)

rev_cleaned.to_csv(
    "project_files/rev_cleaned.csv",
    index=False,
    encoding="utf-8"
)

anzahl_neu_berechnet = mask.sum()
print("Neu berechnete gesamt_note:", anzahl_neu_berechnet)
rev_cleaned.dtypes

## Doc nach Ärzte gruppieren

In [ ]:
# Gruppierung Ärzte --> dafür erst Ausgabe Ärzte
doc[["fachrichtung"]].drop_duplicates().sort_values("fachrichtung")

doc_extended = doc.copy()

In [ ]:
#map regeln

def map_fachgruppe(fg):

    # Hausarzt
    if fg in [
        "Allgemeinmediziner",
        "Praktischer Arzt",
        "Innere- & Allgemeinmediziner"
    ]:
        return "Hausärztliche / Allgemeine Versorgung"

    # internistisch
    if fg in [
        "Arbeitsmediziner",
        "Anästhesiologe",
        "Neurologe",
        "Psychiater & Psychotherapie",
        "Facharzt für Psychiatrie & Psychotherapie",
        "Psychosomatische Medizin & Psychotherapie",
        "Facharzt für Psychosom. Medizin & Psychotherapie",
        "Kinder- & Jugendpsychiater",
        "Physikal. & Rehabilit. Mediziner",
        "Strahlentherapeut",
        "Radiologe",
        "Sprach-, Stimm- und kindliche Hörstörungen",
        "Internist"
    ]:
        return "Internistische & konservative Fachärzte"

    # chirurgie
    if fg in [
        "Facharzt für Allgemeinchirurgie",
        "Orthopäde",
        "Orthopäde & Unfallchirurg",
        "Kinderchirurg",
        "Plastischer & Ästhetischer Chirurg",
        "Mund-, Kiefer-, Gesichtschirurg",
        "Urologe"
    ]:
        return "Operative / chirurgische Fachärzte (Humanmedizin)"

    # frauen, kinder & jugend
    if fg in [
        "Frauenarzt (Gynäkologe)",
        "Kinderarzt",
        "Kinder- und Jugendlichenpsychotherapeut"
    ]:
        return "Frauen-, Kinder- & Jugendmedizin"

    # augen, HNO & Haut
    if fg in [
        "Augenarzt",
        "Hals- Nasen- Ohrenarzt",
        "Hautarzt (Dermatologe)"
    ]:
        return "Augen-, HNO- & Hautärzte"

    # Zahnmedizina llgemein
    if fg in [
        "Zahnarzt mit Schwerpunkt Naturheilwesen",
        "Ästhetische Zahnmedizin"
    ]:
        return "Zahnmedizin – Allgemein"

    # Zahnmedizin spezial
    if fg in [
        "Kieferorthopädie",
        "Fachzahnarzt für Kieferorthopädie",
        "Oralchirurgie",
        "Fachzahnarzt für Oralchirurgie",
        "Implantologie",
        "Endodontologie",
        "Parodontologie",
        "Kinderzahnheilkunde",
        "Laserzahnmedizin"
    ]:
        return "Zahnmedizin – Spezialisierungen"

    # keine Fachrichtung
    if fg in [
        "Arzt ohne nähere Fachgebietsangabe",
        "unbekannt"
    ]:
        return "Sonstige / nicht näher spezifiziert"

    # Rest 
    return "Nicht zugeordnet"

doc_extended["fachgruppe"] = doc_extended["fachrichtung"].map(map_fachgruppe)

nicht_zugeordnet = (
    doc_extended.loc[
        doc_extended["fachgruppe"] == "Nicht zugeordnet",
        "fachrichtung"
    ]
    .dropna()
    .unique()
)

print("Nicht zugeordnete Fachrichtungen:")
print(nicht_zugeordnet)

## doc nach Region sortieren

In [ ]:
#Region zusammenlegen
staedte_doc = (
    doc["stadt"]
    .dropna().drop_duplicates().to_frame(name="stadt")
)

staedte_doc

In [ ]:
def normalize_city_consistent(city):
    if pd.isna(city):
        return None

    city = city.lower().strip()

    city = (
        city.replace("ä", "ae")
            .replace("ö", "oe")
            .replace("ü", "ue")
            .replace("ß", "ss")
    )

    # Zusätze entfernen
    city = re.sub(r",.*$", "", city)           # alles nach Komma
    city = re.sub(r"\(.*?\)", "", city)        # Klammern
    city = re.sub(r"\s+bei\s+.*$", "", city)   # "bei XY"

    # Fluss-/Regionszusätze
    city = re.sub(r"\s+a\.?\s*[a-z]+.*$", "", city)
    city = re.sub(r"\s+am\s+.*$", "", city)
    city = re.sub(r"\s+an\s+der\s+.*$", "", city)
    city = re.sub(r"\s+im\s+.*$", "", city)

    # Titel entfernen
    city = re.sub(
        r"\b(stadt|st|kreisstadt|kupferstadt|bluetenstadt|"
        r"hansestadt|universitaetsstadt|wissenschaftsstadt)\b",
        "",
        city
    )

    # Großstädte
    if city.startswith("frankfurt"):
        city = "frankfurt"
    elif city.startswith("muenchen"):
        city = "muenchen"
    elif city.startswith("berlin"):
        city = "berlin"
    elif city.startswith("koeln"):
        city = "koeln"
    elif city.startswith("muelheim"):
        city = "muelheim"
    elif city.startswith("bad homburg"):
        city = "bad homburg"

    city = re.sub(r"\s+", " ", city).strip()
    return city

In [ ]:
# Bundesland
bl = (
    pd.read_csv("project_files/bundesland.csv")
      [["Ort", "Bundesland"]]
      .drop_duplicates()
      .rename(columns={"Ort": "stadt", "Bundesland": "bundesland"})
)

bl["stadt_norm"] = bl["stadt"].apply(normalize_city_consistent)


# Einwohner / Fläche
einw = pd.read_csv(
    "project_files/staedte_einw.csv",
    sep=";",
    encoding="latin1",
    names=["stadt", "plz", "flaeche_km2", "einwohner"],
    header=0
)

einw["einwohner"] = (
    einw["einwohner"].astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(" ", "", regex=False)
)
einw["einwohner"] = pd.to_numeric(einw["einwohner"], errors="coerce")

einw["flaeche_km2"] = (
    einw["flaeche_km2"].astype(str)
    .str.replace(",", ".", regex=False)
)
einw["flaeche_km2"] = pd.to_numeric(einw["flaeche_km2"], errors="coerce")

einw["stadt_norm"] = einw["stadt"].apply(normalize_city_consistent)
einw = einw[["stadt_norm", "flaeche_km2", "einwohner"]]

In [ ]:
doc_extended["stadt_norm"] = doc_extended["stadt"].apply(normalize_city_consistent)

# Bundesland
doc_extended = doc_extended.merge(
    bl[["stadt_norm", "bundesland"]],
    on="stadt_norm",
    how="left"
)

# Einwohner + Fläche
doc_extended = doc_extended.merge(
    einw,
    on="stadt_norm",
    how="left"
)

# Aufräumen
doc_extended = doc_extended.drop(columns=["stadt_norm"])

In [ ]:
manual_bundesland_map = {
    # Brandenburg
    "ahrensfelde": "Brandenburg",
    "bad freienwalde": "Brandenburg",
    "bad saarow": "Brandenburg",
    "buckow": "Brandenburg",
    "eggersdorf": "Brandenburg",
    "eichwalde": "Brandenburg",
    "erkner": "Brandenburg",
    "fredersdorf-vogelsdorf": "Brandenburg",
    "fuerstenwalde": "Brandenburg",
    "glienicke": "Brandenburg",
    "hoppegarten": "Brandenburg",
    "kleinmachnow": "Brandenburg",
    "neuenhagen": "Brandenburg",
    "rehfelde": "Brandenburg",
    "ruedersdorf": "Brandenburg",
    "schoeneiche": "Brandenburg",
    "schildow": "Brandenburg",
    "seelow": "Brandenburg",
    "storkow": "Brandenburg",

    # Rheinland-Pfalz
    "adenau": "Rheinland-Pfalz",
    "altenahr": "Rheinland-Pfalz",
    "alzey": "Rheinland-Pfalz",
    "antweiler": "Rheinland-Pfalz",
    "bad duerkheim": "Rheinland-Pfalz",
    "darscheid": "Rheinland-Pfalz",
    "kaisersesch": "Rheinland-Pfalz",
    "landau": "Rheinland-Pfalz",
    "linz": "Rheinland-Pfalz",
    "mayen": "Rheinland-Pfalz",
    "pruem": "Rheinland-Pfalz",

    # Hessen
    "bad hersfeld": "Hessen",
    "bad homburg": "Hessen",
    "kronberg": "Hessen",

    # NRW
    "gangelt": "Nordrhein-Westfalen",
    "heinsberg": "Nordrhein-Westfalen",
    "muelheim": "Nordrhein-Westfalen",

    # Baden-Württemberg
    "freiburg": "Baden-Württemberg",
    "gundelfingen": "Baden-Württemberg",

    # Bayern
    "eichenau": "Bayern",
    "haar": "Bayern",
    "kempten": "Bayern",
    "markt schwaben": "Bayern",
    "pullach": "Bayern",

    # Sachsen-Anhalt
    "muecheln": "Sachsen-Anhalt",

    # Sachsen
    "zschopau": "Sachsen",

    # Schleswig-Holstein
    "bad bramstedt": "Schleswig-Holstein",
    "schleswig": "Schleswig-Holstein"
}

mask = doc_extended["bundesland"].isna()
doc_extended.loc[mask, "bundesland"] = (
    doc_extended.loc[mask, "stadt"]
    .apply(normalize_city_consistent)
    .map(manual_bundesland_map)
)

In [ ]:
print("Fehlende Bundesländer:", doc_extended["bundesland"].isna().sum())
print("Fehlende Einwohner:", doc_extended["einwohner"].isna().sum())
print("Fehlende Fläche:", doc_extended["flaeche_km2"].isna().sum())

## Bewertungen pro Arzt zählen

In [ ]:
#bewertungen zählen für Übersicht

bewertungen_pro_arzt = (
    rev_cleaned
    .groupby("ref_id")
    .size()
    .reset_index(name="anzahl_bewertungen")
)

bewertungen_pro_arzt["anzahl_bewertungen"] = (
    bewertungen_pro_arzt["anzahl_bewertungen"].fillna(0).astype(int)
)

(bewertungen_pro_arzt["anzahl_bewertungen"] == 0).sum()
bewertungen_pro_arzt.head()

In [ ]:
#doc_extended erstellen um alles drin zu haben
doc_extended = doc_extended.drop(
    columns=["anzahl_bewertungen"],
    errors="ignore"
)

doc_extended = doc_extended.merge(
    bewertungen_pro_arzt,
    left_on="arzt_id",
    right_on="ref_id",
    how="left"
).drop(columns=["ref_id"])


doc_extended["anzahl_bewertungen"] = (
    doc_extended["anzahl_bewertungen"]
    .fillna(0)
    .astype(int)
)

doc_extended.to_csv(
    "project_files/doc_extended.csv",
    sep=";",
    index=False,
    encoding="utf-8"
)

# Durchschnittsbewertung pro Arzt rausfinden                                     

In [ ]:
doc_extended = pd.read_csv(
    "project_files/doc_extended.csv",
    sep=";",
    encoding="utf-8"
)

durchschnitt_pro_arzt = (
    rev_cleaned
    .groupby("ref_id")["gesamt_note"]
    .mean()
    .reset_index(name="durchschnittsbewertung")
)

durchschnitt_pro_arzt.head()

doc_extended = doc_extended.merge(
    durchschnitt_pro_arzt,
    left_on="arzt_id",
    right_on="ref_id",
    how="left"
)

doc_extended.columns

doc_extended = doc_extended.drop(columns=["ref_id"])

doc_extended["durchschnittsbewertung"] = (
    doc_extended["durchschnittsbewertung"]
    .round(2)
)

print("Ärzte ohne Bewertungen:")
print(
    doc_extended.loc[
        doc_extended["durchschnittsbewertung"].isna(),
        ["arzt_id", "stadt"]
    ].head()
)

print("\nBeispiel:")
print(
    doc_extended[[
        "arzt_id",
        "durchschnittsbewertung"
    ]].drop_duplicates().head(10)
)

doc_extended.to_csv(
    "project_files/doc_extended.csv",
    sep=";",
    index=False,
    encoding="utf-8"
)

In [ ]:
doc_extended.head()
doc_extended.columns